# dataclasses-replace-args — worked example 3: Derive a config in two stages with chained replace calls

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclasses-replace-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Because `dataclasses.replace` returns a fresh dataclass, its result can itself be fed into another `replace`. Chaining lets you derive a config in stages — e.g. first apply a shared 'debug' profile, then apply a per-run override — without ever touching the original base. Each stage produces a new immutable object.

## Worked solution

**Goal:** from one base, first apply a shared debug profile (`epochs=1, batch_size=4`), then apply a per-run override on top.

1. **Stage one: the profile.** `debug = replace(base, **profile)` clones base with the profile fields set. `base` is untouched.
2. **Stage two: per-run on top.** `replace(debug, **run_override)` starts from the *debug* clone, not base, so it inherits the profile's `epochs=1` and `batch_size=4` and only changes what the run override names.
3. **Why chaining is safe.** Each `replace` returns a new object, so stage two reading from `debug` never corrupts base or the profile values you set in stage one.
4. **Field resolution order.** A field's final value is whatever the *last* stage that mentioned it set; fields no stage touched fall back through to base. Here the run override sets `lr`, so lr wins from stage two while `epochs`/`batch_size` win from stage one and `optimizer_name` falls all the way back to base.

The output prints the final derived config, showing profile fields and the run override both landed while base stayed default.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def derive_config(base, profile, run_override):
    debug = replace(base, **profile)
    return replace(debug, **run_override)

base = TrainingArgs()
final = derive_config(base, {'epochs': 1, 'batch_size': 4}, {'lr': 2e-4})
print('final:', final.lr, final.batch_size, final.epochs, final.optimizer_name)
print('base unchanged:', base.lr, base.batch_size, base.epochs)